# 01 — RF再現とGaussian Process Regression

受領した3つのCSVを使い、(1) R `randomForest` + `pls` の再現、(2) 同じ固定10-foldでGPRを比較します。

重要: 標準化、角度変換、分散ゼロ列除去、`X_proc` のPCAは各訓練foldだけでfitします。

In [ ]:
# GitHubから開いたColabはファイルを自動取得しないため、最初にcloneします。
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/futoshi-futami/Chemistory.git'
cwd = Path.cwd()
if (cwd / 'pyproject.toml').exists():
    PROJECT_ROOT = cwd
elif (Path('/content/Chemistory') / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/Chemistory')
elif 'google.colab' in sys.modules:
    PROJECT_ROOT = Path('/content/Chemistory')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    raise FileNotFoundError('Chemistory project rootでノートブックを実行してください。')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Python環境
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

## A. 入力データの整合性検査

In [ ]:
from chemistory_gpr.handoff import load_handoff_data

DATA_DIR = PROJECT_ROOT / 'data' / 'gpr_handoff'
data = load_handoff_data(DATA_DIR)
print('n =', len(data.y))
print('base feature count =', data.base.shape[1] - 2)
print('X_proc feature count =', data.xproc.shape[1] - 1)
print('fold counts =', pd.Series(data.fold_id).value_counts().sort_index().to_dict())
display(pd.read_csv(DATA_DIR / '04_reference_RF_results.csv'))

## B. R randomForestの再現

RFはRの `randomForest` と `pls::plsr(method='simpls')` をそのまま使います。R 3.6で `sample()` の方式が変わったため、現在の `Rejection` と旧 `Rounding` を両方試します。

In [ ]:
import shutil

IN_COLAB = 'google.colab' in sys.modules
if shutil.which('Rscript') is None and IN_COLAB:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-base', 'r-cran-randomforest', 'r-cran-pls'], check=True)
elif shutil.which('Rscript') is None:
    print('この環境にはRscriptがないためRFセルだけをスキップします。Colabでは自動導入されます。')
else:
    package_check = 'missing <- c("randomForest","pls")[!sapply(c("randomForest","pls"), requireNamespace, quietly=TRUE)]; if(length(missing)) install.packages(missing, repos="https://cloud.r-project.org")'
    subprocess.run(['Rscript', '-e', package_check], check=True)
print('Rscript =', shutil.which('Rscript'))

In [ ]:
if shutil.which('Rscript'):
    subprocess.run([sys.executable, 'scripts/run_rf_reproduction.py'], check=True)
    rf_metrics = pd.read_csv(RESULTS / 'rf_reproduction_all_metrics.csv')
    rf_comparison = pd.read_csv(RESULTS / 'rf_reproduction_comparison.csv')
    display(rf_metrics)
    display(rf_comparison)
else:
    print('RF再現は未実行です。')

## C. GPR候補の固定10-fold比較

RBFではなくMatérn kernelを用い、baseのみ、周期角度、`X_proc` fold内PCAを比較します。`R2` は相関係数の二乗ではなく、`1-SSE/SST` です。

In [ ]:
from chemistory_gpr.handoff import cross_validate_handoff, default_handoff_candidates

candidate_metrics = []
candidate_predictions = {}
for config in default_handoff_candidates():
    print('Running', config.name)
    prediction, metrics = cross_validate_handoff(data, config)
    candidate_predictions[config.name] = prediction
    metrics = {k: v for k, v in metrics.items() if k != 'kernels'}
    candidate_metrics.append(metrics)
    prediction.to_csv(RESULTS / f'gpr_handoff_oof_{config.name}.csv', index=False)
metric_table = pd.DataFrame(candidate_metrics).sort_values('R2', ascending=False)
metric_table.to_csv(RESULTS / 'gpr_handoff_metrics.csv', index=False)
display(metric_table[['model','R2','RMSE','MAE','coverage_95','width_95','NLPD']])

In [ ]:
# 最良候補のOOF予測と不確実性
best_name = metric_table.iloc[0]['model']
best = candidate_predictions[best_name]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].errorbar(best['y'], best['pred_mean'], yerr=1.96*best['pred_std'], fmt='o', ms=4, alpha=.65)
limits = [min(best['y'].min(), best['lower_95'].min()), max(best['y'].max(), best['upper_95'].max())]
axes[0].plot(limits, limits, '--', color='black')
axes[0].set(xlabel='Observed y', ylabel='OOF predicted y', title=f'{best_name}: mean ± 95%')
ordered = best.sort_values('pred_std').reset_index(drop=True)
axes[1].fill_between(np.arange(len(ordered)), ordered['lower_95'], ordered['upper_95'], alpha=.25, label='95% interval')
axes[1].plot(ordered['y'].to_numpy(), '.', ms=3, label='observed')
axes[1].plot(ordered['pred_mean'].to_numpy(), '-', lw=1, label='mean')
axes[1].set(title='OOF predictive intervals (sorted by uncertainty)', xlabel='sample', ylabel='y')
axes[1].legend()
plt.tight_layout(); plt.show()

### 読み方

既定データでは、`X_proc`をPCA8にして結合したMatérn GPRが最も良好です。95% coverageが名目値に近いかも同時に確認してください。候補選択自体も同じCV結果を見ているため、最終的な外部評価には新しい独立データが必要です。